In [1]:
import torch
import qewton
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

In [2]:
input_data = torch.load("../data/integrator/f_data.pt", weights_only=True)
output_data = torch.load("../data/integrator/u_data.pt", weights_only=True)

train_in_data, train_out_data = input_data[:4000], output_data[:4000]
test_in_data, test_out_data = input_data[4000:], output_data[4000:]

batch_size = 1000


In [3]:
F = qewton.config.Variable("f", 1)
X = qewton.config.Variable("x", 1)
U = qewton.config.Variable("u", 1)

dataset = qewton.data.GridDataSet(
    data=[train_in_data, train_out_data], 
    feature_variables=[F, U],
)
test_dataset = qewton.data.GridDataSet(
    data=[test_in_data, test_out_data],
    feature_variables=[F, U],
)

In [4]:
data_loader = qewton.data.DataLoader(
    data_set=dataset,
    test_data_set=test_dataset,
    batch_size=batch_size,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

qewton.visualization.Figure(data_loader.visualize(mode=qewton.optim.EvaluationPhase.TRAIN)).show()

In [5]:
model = qewton.algorithms.PCANet(
    input_variable=F,
    output_variable=U,
    pca_n_input=20,
    pca_n_output=20,
    data_source_node=data_loader,
    fcn_hidden_layers=2,
    fcn_hidden_neurons=50,
)

In [6]:
constraint = qewton.constraints.MSEConstraint()
computation_graph = qewton.Graph()
computation_graph.connect(data_loader.get_output_port(F), model.input_ports[0])
computation_graph.connect(data_loader.get_output_port(U), model.input_ports[1])
computation_graph.connect(model, constraint.input_1)
computation_graph.connect(data_loader.get_output_port(U), constraint.input_2)

In [7]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_graph],
    training_objectives=[constraint],
    device="cuda:0",
)

trainer.run()

Optimization Phase 1:  19%|█▊        | 463/2500 [00:01<00:05, 387.70it/s, loss=0.00467]

Optimization Phase 1: 100%|██████████| 2500/2500 [00:08<00:00, 306.85it/s, loss=0.000389]


In [ ]:
layout = computation_graph.visualize(
    [model.output_ports[0], data_loader.get_output_port(F), data_loader.get_output_port(U)],
    mode=qewton.optim.EvaluationPhase.TRAIN,
    device="cuda:0",
)
fig = qewton.visualization.Figure(layout) #.show()
app = qewton.viz.DashApplication.create()